# 03 Double descent with random features

Lecture 10, training and generalization. Book chapter 7 (memorization and
double descent).

**Goals**

- Fit a random-features model (a frozen random ReLU layer, then a linear
  readout) with the *minimum-norm* least-squares solution, for a growing
  number of features $p$ and a fixed number of training samples $n$.
- Observe the test error as $p$ crosses the interpolation threshold $p = n$.

**Expected finding** (asserted in `tests/test_examples.py::test_03_*`): the
test error first follows the classical U-curve, then *peaks* at $p = n$,
where the model just interpolates the noisy training data, and *decreases
again* for $p > n$, to below the best under-parametrized error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glassnn import Tensor, backend, manual_seed, nn, no_grad

backend.set_default_dtype("float64")

## A sparse linear signal

$y = x^\top \beta + \varepsilon$ with $x \in \mathbb R^{20}$, five non-zero
coefficients and noise $\varepsilon \sim \mathcal N(0, 0.1^2)$; $n = 100$
training samples and 2000 test samples.

In [ ]:
def sparse_regression(n, seed, d=20, k=5, noise=0.1):
    rng = np.random.default_rng(seed)
    beta = np.zeros(d)
    beta[:k] = 1 / np.sqrt(k)
    X = rng.normal(size=(n, d))
    return X, X @ beta + noise * rng.normal(size=n)


N_TRAIN = 100
X_train, y_train = sparse_regression(N_TRAIN, seed=0)
X_test, y_test = sparse_regression(2000, seed=1)

## Random ReLU features and the minimum-norm readout

The features $\phi(x) = \operatorname{relu}(W x + b) \in \mathbb R^p$ come
from a GlassNN `Linear` layer that is never trained (random features,
Rahimi and Recht 2007). The readout $\theta$ minimizes
$\lVert \Phi \theta - y \rVert^2$; when $p \ge n$ there are infinitely many
interpolating solutions, and the pseudo-inverse picks the one with the
smallest $\lVert \theta \rVert$. Gradient descent from $\theta = 0$
converges to the same solution.

In [ ]:
def random_features_fit(p, seed):
    manual_seed(seed)
    features = nn.Sequential(nn.Linear(X_train.shape[1], p), nn.ReLU())
    with no_grad():
        phi_train = features(Tensor(X_train)).data
        phi_test = features(Tensor(X_test)).data
    theta = np.linalg.pinv(phi_train) @ y_train
    return {
        "train": np.mean((phi_train @ theta - y_train) ** 2),
        "test": np.mean((phi_test @ theta - y_test) ** 2),
        "norm": np.linalg.norm(theta),
    }


widths = np.unique(np.r_[np.geomspace(5, 5000, 31).round().astype(int), N_TRAIN])
results = {}
for p in widths:
    fits = [random_features_fit(p, seed) for seed in range(10)]
    results[p] = {key: np.median([fit[key] for fit in fits]) for key in fits[0]}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key, style in [("test", "o-"), ("train", "s--")]:
    axes[0].loglog(
        widths,
        [results[p][key] + 1e-16 for p in widths],
        style,
        markersize=3,
        label=f"{key} error",
    )
axes[0].set_ylim(1e-3, 1e3)
axes[1].loglog(widths, [results[p]["norm"] for p in widths], "o-", markersize=3)
axes[1].set_ylabel(r"$\|\theta\|$ of the minimum-norm solution")
for ax in axes:
    ax.axvline(N_TRAIN, color="gray", linestyle=":")
    ax.set_xlabel("number of random features $p$")
axes[0].set_ylabel("mean squared error (median of 10 seeds)")
axes[0].legend()
plt.show()

In [ ]:
best_under = min(results[p]["test"] for p in widths if p < N_TRAIN)
print(f"best test error with p < n:   {best_under:.3f}")
print(f"test error at p = n = {N_TRAIN}:     {results[N_TRAIN]['test']:.1f}")
print(f"test error at p = {widths[-1]}:      {results[widths[-1]]['test']:.3f}")

**Finding.** The test error peaks sharply at $p = n$, the interpolation
threshold, and then decreases again; with many more features than samples
the interpolating model generalizes better than any under-parametrized one.
The right panel explains the peak: at $p = n$ the system
$\Phi \theta = y$ has exactly one solution, and fitting the noise needs a
huge $\lVert\theta\rVert$ (the smallest singular value of $\Phi$ is close to
0). For $p \gg n$ there are many interpolating solutions, and the
minimum-norm one spreads the noise over many directions where it does
little harm at test time (Belkin et al. 2019; Nakkiran et al. 2020;
Bartlett et al. 2020).

**Exercises.**

1. Set the noise to 0. What happens to the peak?
2. Add a ridge penalty, $\theta = (\Phi^\top\Phi + \lambda I)^{-1}\Phi^\top y$,
   for a few values of $\lambda$. What does it do to the peak?
3. Fix $p = 200$ and vary $n$ instead: is more data always better?